In [ ]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier, log_evaluation, early_stopping
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import optuna
import warnings
warnings.filterwarnings('ignore')
# Load the dataset
data_path = '../data/processed/'
data      = pd.read_parquet(data_path + 'train_data.parquet')
data.head()

In [ ]:
# #baseline model
X = data.drop(columns=['TARGET', 'SK_ID_CURR'])
y = data['TARGET']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
# model = LGBMClassifier(
#         n_estimators=1000,
#         learning_rate=0.05,
#         num_leaves=31,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         n_jobs=-1,
#         random_state=42
#     )
# model.fit(X_train, y_train) 

In [ ]:
# y_pred = model.predict(X_val)
# print(classification_report(y_val, y_pred))
# print(confusion_matrix(y_val, y_pred))
# y_pred_proba = model.predict_proba(X_val)[:, 1]
# print(f'ROC AUC Score: {roc_auc_score(y_val, y_pred_proba)}')

In [ ]:
# import time
# from sklearn.model_selection import StratifiedKFold

# def objective(trial):
#     params = {
#         'objective': 'binary',
#         'metric': 'auc',
#         'n_jobs': -1,
#         'verbosity':-1,
#         'random_state': 42,
#         'device_type': 'gpu',  
#         'gpu_platform_id': 0,  
#         'gpu_device_id': 0,
#         'n_estimators': 10000,
#         'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),

#         'num_leaves': trial.suggest_int('num_leaves', 20, 150),
#         'max_depth': trial.suggest_int('max_depth', 3, 12),
#         'min_child_samples': trial.suggest_int('min_child_samples', 100, 2000),

#         'subsample': trial.suggest_float('subsample', 0.5, 1.0),
#         'subsample_freq': trial.suggest_int('subsample_freq', 1, 10),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),

#         'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
#         'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),

#         'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 15.0)
#     }

#     start_trial_time = time.time()
#     cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#     cv_scores = []

#     for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
#         # Chia data cho fold hiện tại
#         X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
#         X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

#         model = LGBMClassifier(**params)
#         model.fit(
#             X_tr, y_tr,
#             eval_set=[(X_va, y_va)],
#             callbacks=[
#                 early_stopping(stopping_rounds=100, verbose=False),
#                 log_evaluation(0)
#             ]
#         )

#         y_pred_proba = model.predict_proba(X_va)[:, 1]
#         auc = roc_auc_score(y_va, y_pred_proba)
#         cv_scores.append(auc)

#     mean_auc    = np.mean(cv_scores)
#     std_auc     = np.std(cv_scores)
#     trial_time  = time.time() - start_trial_time
#     print(f"--> Time: {trial_time:.1f}s | Mean AUC: {mean_auc:.4f} (+/- {std_auc:.4f})")

#     return mean_auc
# warnings.filterwarnings('ignore')
# n_trials = 30

# total_start_time = time.time()
# optuna.logging.set_verbosity(optuna.logging.WARNING)
# study = optuna.create_study(direction='maximize')
# study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

# total_time_mins = (time.time() - total_start_time) / 60
# print("\n" + "="*50)
# print(f"Hoàn thành Tuning trong {total_time_mins:.2f} phút!")
# print(f"Best Mean 5-Fold AUC: {study.best_value:.4f}") 
# for key, value in study.best_params.items():  
#     print(f"    '{key}': {value},")

In [ ]:
best_params = {
    'learning_rate': 0.007,
    'num_leaves': 70,
    'max_depth': 12,
    'min_child_samples': 1290,
    'subsample': 0.85,
    'subsample_freq': 1,
    'colsample_bytree': 0.54,
    'reg_alpha': 0.001,
    'reg_lambda': 0.06,
    'scale_pos_weight': 7.7
}

In [ ]:
best_model = LGBMClassifier(n_estimators=10000, random_state=42, **best_params)
best_model.fit(X_train, y_train)
y_train_pred = best_model.predict_proba(X_train)[:, 1]
y_pred_proba = best_model.predict_proba(X_val)[:, 1]
print(f"ROC AUC Score train: {roc_auc_score(y_train, y_train_pred)}")
print(f'ROC AUC Score: {roc_auc_score(y_val, y_pred_proba)}')

In [ ]:
# train on full data
best_model.fit(X, y)

In [ ]:
#Save model
import joblib
joblib.dump(best_model, '../models/lgbm_best_model.pkl')

# submission from test data
test_data = pd.read_parquet(data_path + 'test_data.parquet')
X_test = test_data.drop(columns=['SK_ID_CURR'])
test_pred_proba = best_model.predict_proba(X_test)[:, 1]
submission = pd.DataFrame({
    'SK_ID_CURR': test_data['SK_ID_CURR'],
    'TARGET': test_pred_proba
})
submission.to_csv('../data/processed/lgbm_submission.csv', index=False)

In [ ]:
# confusion matrix, classification report
y_predicted = (y_pred_proba >= 0.5).astype(int)
print(classification_report(y_val, y_predicted))
print(confusion_matrix(y_val, y_predicted))
